In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 150)


df = pd.read_csv('Combined_Flights_2019.csv')

In [4]:

def verifier_redondance(df, col_a, col_b):
    n_a = df[col_a].nunique()
    n_b = df[col_b].nunique()
    n_paires = df[[col_a, col_b]].drop_duplicates().shape[0]
    est_1_1 = (n_paires == n_a) and (n_paires == n_b)
    print(f"{col_a} ({n_a}) vs {col_b} ({n_b}) -> {n_paires} paires uniques "
          f"-> {'REDONDANT (1:1)' if est_1_1 else 'PAS 1:1'}")


# ------------------------------------------------------------------
# 1. Operated_or_Branded_Code_Share_Partners
# ------------------------------------------------------------------
print("=== Operated_or_Branded_Code_Share_Partners ===")
print("Nb valeurs uniques :", df['Operated_or_Branded_Code_Share_Partners'].nunique())
print(df['Operated_or_Branded_Code_Share_Partners'].value_counts().head(10))
print()
verifier_redondance(df, 'Operated_or_Branded_Code_Share_Partners', 'IATA_Code_Operating_Airline')
verifier_redondance(df, 'Operated_or_Branded_Code_Share_Partners', 'IATA_Code_Marketing_Airline')
print()

# ------------------------------------------------------------------
# 2. Flight_Number_Marketing_Airline vs Flight_Number_Operating_Airline
# ------------------------------------------------------------------
print("=== Flight_Number_Marketing_Airline vs Operating ===")
verifier_redondance(df, 'Flight_Number_Marketing_Airline', 'Flight_Number_Operating_Airline')
n_diff = (df['Flight_Number_Marketing_Airline'] != df['Flight_Number_Operating_Airline']).sum()
print(f"Nb de lignes où les deux numéros diffèrent : {n_diff:,} "
      f"({n_diff/len(df):.2%})")
print()

# ------------------------------------------------------------------
# 3. DivAirportLandings — n'a de sens que si Diverted=True
# ------------------------------------------------------------------
print("=== DivAirportLandings ===")
print(df['DivAirportLandings'].value_counts())
print()
print("DivAirportLandings > 0 mais Diverted == False :",
      df[(df['DivAirportLandings'] > 0) & (~df['Diverted'])].shape[0])
print("Diverted == True mais DivAirportLandings == 0 :",
      df[df['Diverted'] & (df['DivAirportLandings'] == 0)].shape[0])

=== Operated_or_Branded_Code_Share_Partners ===
Nb valeurs uniques : 15
Operated_or_Branded_Code_Share_Partners
WN              1363946
AA_CODESHARE    1164263
DL               991986
AA               946776
UA_CODESHARE     945494
DL_CODESHARE     774047
UA               625910
B6               297411
AS               264817
NK               204845
Name: count, dtype: int64

Operated_or_Branded_Code_Share_Partners (15) vs IATA_Code_Operating_Airline (26) -> 35 paires uniques -> PAS 1:1
Operated_or_Branded_Code_Share_Partners (15) vs IATA_Code_Marketing_Airline (10) -> 15 paires uniques -> PAS 1:1

=== Flight_Number_Marketing_Airline vs Operating ===
Flight_Number_Marketing_Airline (7130) vs Flight_Number_Operating_Airline (7142) -> 7904 paires uniques -> PAS 1:1
Nb de lignes où les deux numéros diffèrent : 978 (0.01%)

=== DivAirportLandings ===
DivAirportLandings
0    8069685
1      20649
9       1208
2        137
3          5
Name: count, dtype: int64

DivAirportLandings > 0 mais Di

In [5]:


print("Shape avant :", df.shape)

colonnes_redondantes = [
    'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID',
    'OriginCityName', 'OriginStateFips', 'OriginStateName', 'OriginWac',
    'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID',
    'DestCityName', 'DestStateFips', 'DestStateName', 'DestWac',
    'Airline', 'Marketing_Airline_Network', 'DOT_ID_Marketing_Airline',
    'Operating_Airline', 'DOT_ID_Operating_Airline',
    'Tail_Number',
    'DepTimeBlk', 'ArrTimeBlk',
    'DistanceGroup',
    'DepDelayMinutes',
    'Flight_Number_Operating_Airline',
]

colonnes_fuite_atterrissage_supplementaires = ['DivAirportLandings']

colonnes_fuite_apres_decollage = [
    'DepTime', 'DepDelay', 'DepDel15', 'DepartureDelayGroups',
    'TaxiOut', 'WheelsOff',
]

colonnes_fuite_apres_atterrissage = [
    'WheelsOn', 'TaxiIn', 'ArrTime', 'ArrDelay', 'ArrivalDelayGroups',
    'AirTime', 'ActualElapsedTime',
]

colonnes_target_brute = ['ArrDelayMinutes']

df['is_codeshare'] = (
    df['IATA_Code_Marketing_Airline'] != df['IATA_Code_Operating_Airline']
).astype(int)
colonnes_redondantes.append('Operated_or_Branded_Code_Share_Partners')

a_supprimer = (
    colonnes_redondantes
    + colonnes_fuite_apres_decollage
    + colonnes_fuite_apres_atterrissage
    + colonnes_fuite_atterrissage_supplementaires
    + colonnes_target_brute
)

a_supprimer_existantes = [c for c in a_supprimer if c in df.columns]
manquantes = set(a_supprimer) - set(a_supprimer_existantes)
if manquantes:
    print("Colonnes déjà absentes (ignorées) :", manquantes)

df = df.drop(columns=a_supprimer_existantes)

print("Shape après :", df.shape)
print("\nColonnes restantes :")
print(df.columns.tolist())


df.to_csv("Combined_Flights_Clean.csv", index=False)

Shape avant : (8091684, 23)
Colonnes déjà absentes (ignorées) : {'OriginStateFips', 'DepDelayMinutes', 'TaxiOut', 'Airline', 'DOT_ID_Operating_Airline', 'DestCityName', 'WheelsOn', 'ArrTime', 'ActualElapsedTime', 'DepTimeBlk', 'Marketing_Airline_Network', 'OriginCityName', 'Operating_Airline', 'DestAirportSeqID', 'WheelsOff', 'ArrDelay', 'DepDelay', 'ArrivalDelayGroups', 'DestCityMarketID', 'DestStateFips', 'OriginStateName', 'Tail_Number', 'AirTime', 'DOT_ID_Marketing_Airline', 'ArrTimeBlk', 'DepartureDelayGroups', 'DestStateName', 'DestAirportID', 'ArrDelayMinutes', 'OriginAirportID', 'DestWac', 'DepTime', 'TaxiIn', 'DepDel15', 'OriginAirportSeqID', 'DistanceGroup', 'OriginCityMarketID', 'OriginWac'}
Shape après : (8091684, 21)

Colonnes restantes :
['FlightDate', 'Origin', 'Dest', 'Cancelled', 'Diverted', 'CRSDepTime', 'CRSElapsedTime', 'Distance', 'Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'IATA_Code_Marketing_Airline', 'Flight_Number_Marketing_Airline', 'IATA_Code_Oper